# Pipeline Completo de Deteccion de producto faltante

Flujo end-to-end en Colab (GPU) o local (CPU/GPU). La celda 1 resuelve el
entorno y define `ROOT_DIR`; el resto depende unicamente de esa variable.

## Orden de ejecucion

| # | Celda | Accion |
|---|-------|--------|
| 1 | Setup | Descomprime ZIP / clona / resuelve rai­z |
| 2 | Config | Hiperparametros y credenciales AWS |
| 3 | S3 | Descarga del dataset SKU-110K |
| 4 | Limpieza | Verifica imgenes corruptas |
| 5 | YOLO | CSV a formato YOLO |
| 6 | Etiquetas | Heuri­stica geometrica de `missing` |
| 7 | Export | Brazos A (nc=1) y B (nc=2) |
| 8 | Train | Entrenamiento YOLOv8 |
| 9 | Eval | mAP50, mAP50-95, precisipn, recall |
| 10 | Post | Valida heuri­stica antes de la API |

## Colab
1. Subir el ZIP a `/content/`.
2. *Runtime â†’ Change runtime type â†’ GPU*.
3. Ejecutar celdas 1-2, luego 3-10.

## Notas
- Las credenciales AWS se ingresan en la celda 2 en Colab o en `.env` local.
- `download_limit: 200` para pruebas; `0` para el dataset completo.
- Los pesos y artefactos pesados estan en `training/weights/` (gitignored).


In [ ]:
# Celda 1: Setup â€” resuelve ROOT_DIR en Colab o local
import os, subprocess, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT_DIR = Path("/content/repo") if IN_COLAB else None

if IN_COLAB:
    if not (ROOT_DIR / "data_pipeline").is_dir():
        zips = sorted(Path("/content").glob("*.zip"))
        if zips:
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall("/content/_unzip")
            extracted = Path("/content/_unzip/AnyoneAI-object-detection")
            if extracted.is_dir():
                extracted.parent.mkdir(parents=True, exist_ok=True)
                subprocess.run(["mv", str(extracted), str(ROOT_DIR)], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", "dev", "--single-branch",
                            "https://github.com/vchacin/AnyoneAI-object-detection.git",
                            str(ROOT_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ultralytics", "boto3", "tqdm", "pandas", "pillow",
                    "opencv-python-headless", "pyyaml", "matplotlib"], check=True)
else:
    ROOT_DIR = Path.cwd()
    if not (ROOT_DIR / "data_pipeline").is_dir() and \
       (ROOT_DIR.parent / "data_pipeline").is_dir():
        ROOT_DIR = ROOT_DIR.parent

assert (ROOT_DIR / "data_pipeline").is_dir(), f"No se encuentra el repo en {ROOT_DIR}"
for d in ("dataset_sku110k", "configs", "weights", "reports"):
    (ROOT_DIR / "training" / d).mkdir(parents=True, exist_ok=True)
(ROOT_DIR / "data_pipeline" / "dataset_sku110k").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT_DIR))
print("ROOT_DIR =", ROOT_DIR)


In [ ]:
# Celda 2: Configuración — hiperparámetros y credenciales
import json, os, time
from getpass import getpass

if IN_COLAB:
    os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS Access Key ID: ")
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS Secret Access Key:")
else:
    dotenv = ROOT_DIR / ".env"
    if dotenv.is_file():
        for line in dotenv.read_text(encoding="utf-8").splitlines():
            if line.strip() and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                os.environ.setdefault(k.strip(), v.strip().strip("'").strip('"'))

DATA_DIR = ROOT_DIR / "data_pipeline" / "dataset_sku110k"
CONFIGS_DIR = ROOT_DIR / "training" / "configs"
WEIGHTS_DIR = ROOT_DIR / "training" / "weights"
REPORTS_DIR = ROOT_DIR / "training" / "reports"
YOLO_DIR = DATA_DIR / "yolo" / "train"
IMAGES_DIR = DATA_DIR / "images"
ANNOTATIONS_DIR = DATA_DIR / "annotations"
MISSING_OUT = DATA_DIR / "missing_poc"

# Auto-detect: use train CSV if train images exist, otherwise fall back to test
def _detect_split():
    train_imgs = sorted(IMAGES_DIR.glob("train_*.jpg")) if IMAGES_DIR.exists() else []
    test_imgs = sorted(IMAGES_DIR.glob("test_*.jpg")) if IMAGES_DIR.exists() else []
    if train_imgs:
        return "train"
    if test_imgs:
        return "test"
    return "train"

_SPLIT = _detect_split()
CSV_PATH = ANNOTATIONS_DIR / f"annotations_{_SPLIT}.csv"

CFG = {
    "which": "A",
    "epochs": 35,
    "imgsz": 1024,
    "batch": 1,
    "seed": 42,
    "max_det": 750,
    "patience": 100,
    "workers": 0,
    "device": 0 if IN_COLAB else None,
    "base_weights": WEIGHTS_DIR / "yolo26s.pt",
    "coco_weights": WEIGHTS_DIR / "yolo26n.pt",
    "auto_score": 0.5,
    "download_limit": 200,
    "split": _SPLIT,
}
for k, v in CFG.items():
    print(f"{k:14s} = {v}")
print(f"\nCSV_PATH = {CSV_PATH}")
print(f"Split detectado = {_SPLIT} ({'train_*.jpg' if _SPLIT == 'train' else 'test_*.jpg'})")


In [ ]:
# Celda 3: Descargar dataset desde S3 (salta si ya existe localmente)
existing = list(IMAGES_DIR.glob("*.jpg")) if IMAGES_DIR.exists() else []
if existing:
    print(f"Imagenes ya descargadas localmente: {len(existing)}")
    print("Saltando descarga de S3.")
else:
    script = ROOT_DIR / "data_pipeline" / "scripts" / "download_dataset.py"
    cmd = [sys.executable, str(script),
           "--out", str(DATA_DIR), "--bucket", "anyoneai-datasets",
           "--prefix", "SKU-110K/SKU110K_fixed/",
           "--limit", str(CFG["download_limit"])]
    subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
# Celda 4: Verificar integridad de imagenes
script = ROOT_DIR / "data_pipeline" / "scripts" / "clean_images.py"
cmd = [sys.executable, str(script), "--input", str(IMAGES_DIR),
       "--report", str(DATA_DIR / "revision_imagenes.txt")]
subprocess.run(cmd, check=True)


In [ ]:
# Celda 5: Convertir CSV de SKU-110K a formato YOLO
script = ROOT_DIR / "data_pipeline" / "scripts" / "to_yolo_format.py"
cmd = [sys.executable, str(script), "--csv", str(CSV_PATH),
       "--images-dir", str(IMAGES_DIR), "--out", str(YOLO_DIR)]
subprocess.run(cmd, check=True)


In [ ]:
# Celda 6: Etiquetar clase `missing` con heuristica geometrica
import time
script = ROOT_DIR / "labeling" / "generate_missing_labels.py"
coco_weights = CFG["coco_weights"]
cmd = [sys.executable, str(script), "--images-dir", str(IMAGES_DIR),
       "--csv", str(CSV_PATH), "--glob", f"{CFG['split']}_*.jpg",
       "--limit", str(CFG["download_limit"]),
       "--out", str(MISSING_OUT), "--auto-score", str(CFG["auto_score"])]
if not coco_weights.is_file():
    cmd.append("--no-coco")
t0 = time.time()
subprocess.run(cmd, check=True)
elapsed = time.time() - t0
print(f"Generacion de missing: {elapsed:.1f}s")
print(json.dumps(json.loads((MISSING_OUT / "summary.json").read_text()), indent=2))


In [ ]:
# Celda 7: Exportar brazos A (nc=1) y B (nc=2)
script = ROOT_DIR / "labeling" / "export_empty_class" / "export_empty_class.py"
cmd = [sys.executable, str(script), "--yolo-dir", str(YOLO_DIR),
       "--images-dir", str(IMAGES_DIR),
       "--candidates", str(MISSING_OUT / "candidates.csv"), "--arm", "both"]
subprocess.run(cmd, check=True)
for name in ("data_arm_a.yaml", "data_arm_b.yaml"):
    p = CONFIGS_DIR / name
    if p.is_file():
        print(f"--- {name} ---\n{p.read_text()}")


In [ ]:
# Celda 8: Entrenamiento YOLO
from ultralytics import YOLO
import time

which = CFG["which"]
model = YOLO(str(CFG["base_weights"]))

t0 = time.time()
results = model.train(
    data=str(CONFIGS_DIR / f"data_arm_{which.lower()}.yaml"),
    epochs=CFG["epochs"], imgsz=CFG["imgsz"], batch=CFG["batch"],
    seed=CFG["seed"], max_det=CFG["max_det"], patience=CFG["patience"],
    workers=CFG["workers"], project=str(WEIGHTS_DIR),
    name=f"{which}_e{CFG['epochs']}", exist_ok=True,
    **({"device": CFG["device"]} if CFG["device"] is not None else {}),
)
elapsed = time.time() - t0
BEST = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nEntrenamiento completado en {elapsed:.1f}s")
print("best.pt ->", BEST)


In [ ]:
# Celda 9: Evaluacion a mAP50, mAP50-95, precision, recall
test_yaml = CONFIGS_DIR / ("test_obj.yaml" if which == "A" else "test_2cls.yaml")

metrics = YOLO(str(BEST)).val(
    data=str(test_yaml), imgsz=CFG["imgsz"], batch=1,
    max_det=CFG["max_det"], project=str(WEIGHTS_DIR),
    name=f"{which}_e{CFG['epochs']}_val", exist_ok=True,
)

box = metrics.box
summary = {
    "weights": str(BEST),
    "results": {k: round(float(v), 5) for k, v in metrics.results_dict.items()},
    "per_class": {
        "names": list(box.names.values()) if hasattr(box, "names") else None,
        "precision": [round(float(v), 5) for v in box.p],
        "recall": [round(float(v), 5) for v in box.r],
        "ap50": [round(float(v), 5) for v in box.ap50],
        "map50_95": [round(float(v), 5) for v in box.maps],
    },
}
print(json.dumps(summary, indent=2))
(REPORTS_DIR / f"{which}_e{CFG['epochs']}_metrics.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8")


In [ ]:
# Celda 10: Validar heuri­stica de huecos sobre predicciones del modelo
import cv2, numpy as np
from api.gap_heuristic import AUTO_SCORE, score_candidates

detector = YOLO(str(BEST))

def predict_gaps(image_path):
    result = detector.predict(str(image_path), conf=0.25, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy().tolist()
    image = cv2.imread(str(image_path))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image, boxes, score_candidates(gray, boxes, auto_score=CFG["auto_score"])

def draw(image, objects, candidates):
    vis = image.copy()
    for b in objects:
        cv2.rectangle(vis, (round(b[0]), round(b[1])), (round(b[2]), round(b[3])), (0, 200, 0), 1)
    for c in candidates:
        x1, y1, x2, y2 = (round(v) for v in c["box"])
        color = (0, 140, 255) if c["auto"] else (0, 0, 255)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis, f'{c["score"]:.2f}', (x1, max(10, y1 - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return vis

for path in sorted(IMAGES_DIR.glob("*.jpg"))[:5]:
    image, objects, candidates = predict_gaps(path)
    accepted = sum(1 for c in candidates if c["auto"])
    print(f"{path.name}: {len(objects)} productos, {len(candidates)} candidatos, {accepted} aceptados")
    cv2.imwrite(str(REPORTS_DIR / f"heuristic_{path.stem}.jpg"), draw(image, objects, candidates))
